# **PAYSIM DATASET**

## **TASK 2.1: Dataset Survey & Validation**

- **Goal:** Decide the official dataset (PaySim vs AMLSim) and confirm the raw data actually contains enough structure (repeated triangles, dense clusters) to make Task 3 (Motif Finding) and Task 4 (LPA) produce non-trivial results later.

- **Input:** Raw PaySim CSV (`PS_..._log.csv`, ~493MB, 11 columns per Kaggle screenshot); AMLSim docs (backup option only, don't need to download unless PaySim fails validation).

- **Method/Notes:**
  - Load a manageable sample (or full file if memory allows) with PySpark.
  - Check schema against Kaggle's documented columns (already confirmed via screenshot: `step, type, amount, nameOrig, oldbalanceOrg, newbalanceOrig, nameDest, oldbalanceDest, newbalanceDest, isFraud, isFlaggedFraud`).
  - Quantify structural signal *before* committing: overlap between `nameOrig` and `nameDest` sets (accounts acting as both sender/receiver — a precondition for cycles to exist at all), frequency of repeated (src,dst) pairs, and fraud-transaction density.
  - This is **not** full graph construction (that's Task 1) — just a lightweight feasibility check.

- **Expected Output:** A short decision note: "PaySim confirmed / rejected as official dataset" + 3–5 sentence justification with numbers (e.g., "% of accounts appear as both sender and receiver", "fraud transaction count", "isFlaggedFraud count").

- **Dependency/Handoff:** Feeds directly into Task 1 (Person 1 + Person 2). Also informs Person 4 (Motif threshold tuning) and Person 5/6 (whether LPA will find meaningful clusters).

- **Teammate Communication:** Post the decision note + key numbers (overlap %, cycle-feasibility result, fraud count) in the group chat as soon as you finish — don't wait to be asked. Tag Person 1 (they're listed as reviewer for this task in the PCCV) and explicitly flag Person 4 and Person 5/6 by name, since your cycle/community feasibility numbers directly change how they'll approach Tasks 3 and 4.


- **Objective:** Confirm PaySim is usable before committing - not just "could cycles theoretically exist" but "do they actually occur," using a direct empirical test rather than a set-overlap proxy. Also confirm fraud signal density matches Kaggle's documented `CASH_OUT`/`TRANSFER` concentration.

- **Method:** Load with explicit schema (no `inferSchema`) → crosstab transaction type against sender/receiver prefixes to read the real topology → confirm node overlap as a precondition check → directly self-join Customer-to-Customer edges to test for 2-hop chains, reciprocal pairs, and closed 3-node cycles, capping each join with `.limit()` so a high-degree hub account can't explode the computation → cross-check fraud rate by type.

In [6]:
import os
import sys
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, DoubleType, ShortType
)


def create_spark_session() -> SparkSession:
    """Local single-node Spark session, tuned for a ~500MB / ~6.3M-row CSV.
    No GraphFrames jar here — that's only needed from Task 2.5 onward.
    shuffle.partitions is kept low (8) because this runs on one laptop,
    not a cluster; the default 200 partitions would just add scheduling
    overhead for a dataset this size.
    """
    return (
        SparkSession.builder
        .appName("GraphGuard-Task2.1-DatasetExploration")
        .master("local[*]")
        .config("spark.driver.memory", "6g")
        .config("spark.driver.maxResultSize", "2g")
        .config("spark.sql.shuffle.partitions", "8")
        .getOrCreate()
    )


def load_raw(spark: SparkSession, path: str):
    """Explicit schema — never let Spark infer types on a file this size
    (it forces a slow double-read and occasionally mis-types columns).
    isFraud/isFlaggedFraud use ShortType since they're binary flags — a
    small memory win at 6.3M rows.
    """
    schema = StructType([
        StructField("step", IntegerType(), False),
        StructField("type", StringType(), False),
        StructField("amount", DoubleType(), False),
        StructField("nameOrig", StringType(), False),
        StructField("oldbalanceOrg", DoubleType(), False),
        StructField("newbalanceOrig", DoubleType(), False),
        StructField("nameDest", StringType(), False),
        StructField("oldbalanceDest", DoubleType(), False),
        StructField("newbalanceDest", DoubleType(), False),
        StructField("isFraud", ShortType(), False),
        StructField("isFlaggedFraud", ShortType(), False),
    ])

    if not os.path.exists(path):
        print(f"[ERROR] Cannot find raw CSV at: {path}")
        print("        Fix the RAW_PATH variable in main() and re-run.")
        sys.exit(1)

    df = spark.read.csv(path, header=True, schema=schema)
    df.cache()
    return df


def section_1_schema_and_volume(df):
    print("\n" + "=" * 60)
    print("1. SCHEMA & VOLUME CHECK")
    print("=" * 60)
    df.printSchema()
    print(f"Total transactions: {df.count():,}")


def section_2_edge_type_crosstab(df):
    """Which sender/receiver prefix combos actually occur, per transaction type?
    This is a better topology signal than raw node counts — it directly shows
    whether traffic is bipartite (C<->M) or has C<->C chains that could cycle.
    """
    print("\n" + "=" * 60)
    print("2. TRANSACTION TYPE x SENDER/RECEIVER PREFIX CROSSTAB")
    print("=" * 60)
    (
        df.withColumn("orig_prefix", F.substring("nameOrig", 1, 1))
          .withColumn("dest_prefix", F.substring("nameDest", 1, 1))
          .groupBy("type", "orig_prefix", "dest_prefix")
          .agg(F.count("*").alias("n_tx"), F.sum("isFraud").alias("fraud_tx"))
          .orderBy(F.col("n_tx").desc())
          .show(20, truncate=False)
    )


def section_3_node_overlap(df):
    """Precondition check: if no account is ever both sender AND receiver,
    the graph is fully bipartite and 3-node cycles are mathematically
    impossible, regardless of anything else."""
    print("\n" + "=" * 60)
    print("3. SENDER/RECEIVER NODE OVERLAP (cycle precondition)")
    print("=" * 60)
    orig_nodes = df.select(F.col("nameOrig").alias("id")).distinct()
    dest_nodes = df.select(F.col("nameDest").alias("id")).distinct()
    n_orig, n_dest = orig_nodes.count(), dest_nodes.count()
    n_overlap = orig_nodes.intersect(dest_nodes).count()
    n_union = n_orig + n_dest - n_overlap

    print(f"Distinct senders:     {n_orig:,}")
    print(f"Distinct receivers:   {n_dest:,}")
    print(f"Overlap (both roles): {n_overlap:,}  ({n_overlap / n_union:.4%} of all unique accounts)")

    if n_overlap == 0:
        print("[!] Graph is fully bipartite — 3-node cycles are IMPOSSIBLE. Reconsider dataset now.")


def section_4_self_loops(df):
    print("\n" + "=" * 60)
    print("4. SELF-LOOP CHECK (nameOrig == nameDest)")
    print("=" * 60)
    n = df.filter(F.col("nameOrig") == F.col("nameDest")).count()
    print(f"Self-loop transactions: {n:,}")


def section_5_cycle_feasibility(df):
    """The real test. Set overlap (section 3) only proves cycles are
    *possible* — this proves whether they *actually occur*.
    Restricted to Customer-to-Customer edges: in PaySim, nameOrig is always
    a customer (merchants never initiate transactions), so C-to-C traffic
    is the only place a closed loop could form.
    """
    print("\n" + "=" * 60)
    print("5. 3-NODE CYCLE FEASIBILITY TEST (empirical)")
    print("=" * 60)

    c2c = (
        df.filter((F.substring("nameOrig", 1, 1) == "C") & (F.substring("nameDest", 1, 1) == "C"))
          .select(F.col("nameOrig").alias("src"), F.col("nameDest").alias("dst"), F.col("amount"))
    )
    print(f"Customer-to-Customer transactions: {c2c.count():,}")

    # 2-hop: A -> B -> C. .limit() before count() caps the join explosion
    # from any hub account without needing to materialize the full result.
    hop2 = (
        c2c.alias("e1").join(
            c2c.alias("e2"),
            (F.col("e1.dst") == F.col("e2.src")) & (F.col("e1.src") != F.col("e2.dst")),
            "inner",
        ).select(F.col("e1.src").alias("A"), F.col("e1.dst").alias("B"), F.col("e2.dst").alias("C"))
    )
    print(f"2-hop chains found (capped at 2000): {hop2.limit(2000).count():,}")

    # Reciprocal pairs: A -> B and B -> A
    reciprocal = (
        c2c.alias("e1").join(
            c2c.alias("e2"),
            (F.col("e1.src") == F.col("e2.dst")) & (F.col("e1.dst") == F.col("e2.src")),
            "inner",
        ).limit(500).count()
    )
    print(f"Reciprocal pairs found (A->B and B->A, capped at 500): {reciprocal:,}")

    # 3-node cycle: A -> B -> C -> A
    cycle_3 = (
        hop2.alias("p").join(
            c2c.alias("e3"),
            (F.col("p.C") == F.col("e3.src")) & (F.col("p.A") == F.col("e3.dst")),
            "inner",
        ).select("p.A", "p.B", "p.C").limit(10)
    )
    found = cycle_3.count()
    print(f"3-node cycles found (sample, capped at 10): {found:,}")
    if found > 0:
        print("[+] CONCLUSION: PaySim DOES contain closed 3-node transaction cycles.")
        cycle_3.show(truncate=False)
    else:
        print("[-] No 3-node cycle found in this probe.")
        print("    -> Flag to Person 4: may need a wider Motif threshold, or Motif Finding")
        print("       may need to run on the full dataset (not a sample) before concluding.")


def section_6_fraud_by_type(df):
    print("\n" + "=" * 60)
    print("6. FRAUD DENSITY BY TRANSACTION TYPE")
    print("=" * 60)
    (
        df.groupBy("type")
          .agg(
              F.count("*").alias("total_tx"),
              F.sum("isFraud").alias("fraud_tx"),
              F.sum("isFlaggedFraud").alias("flagged_tx"),
          )
          .withColumn("fraud_rate_pct", F.round(F.col("fraud_tx") / F.col("total_tx") * 100, 4))
          .orderBy(F.col("fraud_tx").desc())
          .show()
    )


def main():
    spark = create_spark_session()
    spark.sparkContext.setLogLevel("WARN")

    # Adjust to your actual filename
    # Use an absolute raw string path to avoid working directory issues:
    RAW_PATH = r"D:\_Python\Year4_Semester1\Big Data\graphguard_amls_detection\data\raw\PS_20174392719_1491204439457_log.csv"
    
    # Alternatively, auto-detect between project root and notebooks subfolder:
    if not os.path.exists(RAW_PATH):
        RAW_PATH = os.path.abspath(os.path.join(os.getcwd(), "..", "data", "raw", "PS_20174392719_1491204439457_log.csv"))

    df = load_raw(spark, RAW_PATH)

    section_1_schema_and_volume(df)
    section_2_edge_type_crosstab(df)
    section_3_node_overlap(df)
    section_4_self_loops(df)
    section_5_cycle_feasibility(df)
    section_6_fraud_by_type(df)

    print("\n[+] Task 2.1 exploration complete.")
    spark.stop()


if __name__ == "__main__":
    main()


1. SCHEMA & VOLUME CHECK
root
 |-- step: integer (nullable = true)
 |-- type: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- nameOrig: string (nullable = true)
 |-- oldbalanceOrg: double (nullable = true)
 |-- newbalanceOrig: double (nullable = true)
 |-- nameDest: string (nullable = true)
 |-- oldbalanceDest: double (nullable = true)
 |-- newbalanceDest: double (nullable = true)
 |-- isFraud: short (nullable = true)
 |-- isFlaggedFraud: short (nullable = true)

Total transactions: 6,362,620

2. TRANSACTION TYPE x SENDER/RECEIVER PREFIX CROSSTAB
+--------+-----------+-----------+-------+--------+
|type    |orig_prefix|dest_prefix|n_tx   |fraud_tx|
+--------+-----------+-----------+-------+--------+
|CASH_OUT|C          |C          |2237500|4116    |
|PAYMENT |C          |M          |2151495|0       |
|CASH_IN |C          |C          |1399284|0       |
|TRANSFER|C          |C          |532909 |4097    |
|DEBIT   |C          |C          |41432  |0       |
+--------+

## **Executive Summary Metrics**

- **Total Transactions:** 6,362,620
- **Unique Senders (nameOrig):** 6,353,307 (70.02%)
- **Unique Receivers (nameDest):** 2,722,362 (30.00%)
- **Overlapping Accounts:** 1,769 (0.0195%)
- **Total Unique Graph Vertices (V_total):** 9,073,900


### **1. Schema & Data Volume Verification**

* **Observed Count:** **6,362,620** records ingested across all 11 features without missing rows or casting errors.
* **Schema Integrity:** All features loaded cleanly according to the explicit `StructType`. 
  * *Technical note on Spark CSV behavior:* In Spark's schema printout, columns display as `nullable = true` despite being declared `nullable = False` in the code. This is expected Spark behavior: Spark's native CSV reader does not enforce binary-level non-null constraints upon ingestion because CSV files cannot guarantee constraint compliance at the storage layer. All values are present and clean.
* **Volume Evaluation:** The data size satisfies the Big Data course requirements, providing ~6.36M edges to demonstrate distributed graph partitioning (Vertex Cut vs. Edge Cut), Catalyst query planning, and Tungsten execution performance.

### **2. Transaction Topology & Entity Dynamics**

| Transaction Type | Sender Prefix | Receiver Prefix | Transaction Count ($n$) | Fraud Count | Topological Behavior |
| :--- | :---: | :---: | :---: | :---: | :--- |
| **CASH_OUT** | `C` | `C` | **2,237,500** | **4,116** | Peer-to-Peer / Cash Agent liquidation |
| **PAYMENT** | `C` | `M` | **2,151,495** | **0** | Consumer-to-Merchant (Pure sink edges) |
| **CASH_IN** | `C` | `C` | **1,399,284** | **0** | Capital injection / Inbound liquidity |
| **TRANSFER** | `C` | `C` | **532,909** | **4,097** | P2P fund movement (Layering phase) |
| **DEBIT** | `C` | `C` | **41,432** | **0** | Account adjustment |

* **Key Finding 1 - Asymmetric Initiation:** In PaySim, **100% of transactions originate from Customer accounts (`C`)**. Merchant accounts (`M`) appear exclusively as destination nodes in `PAYMENT` transactions and never initiate transfers ($Out\text{-}Degree(M) = 0$).
* **Key Finding 2 - Strict Structural Sinks:** Merchants operate as dead-end sinks in the directed network. No cycles or onward paths can traverse through a merchant account.

### **3. Graph Bipartiteness & Node Overlap (Precondition for Cycles)**

| Metric | Node Count | Percentage of Total Network ($V_{total}$) |
| :--- | :--- | :--- |
| **Distinct Senders ($V_{orig}$)** | **6,353,307** | **70.02%** |
| **Distinct Receivers ($V_{dest}$)** | **2,722,362** | **30.00%** |
| **Overlapping Nodes ($V_{orig} \cap V_{dest}$)** | **1,769** | **0.0195%** |
| **Total Unique Graph Vertices ($V_{total}$)** | **9,073,900** | **100.00%** |

*(Note: Sender % + Receiver % exceeds 100% by exactly the 0.0195% overlap, reflecting dual-role accounts counted in both sets).*

* **Key Finding:** The graph is **not purely bipartite**. While 99.98% of accounts play a fixed single role (either purely sending or purely receiving), there are **1,769 accounts** that act as both senders and receivers.
* **Topological Role:** These 1,769 accounts are the **structural bridges (routing hubs)** of the entire graph, making multi-hop directed paths ($A \to B \to C$) possible.

### **4. Self-Loop Audit**

* **Observed Count:** **0 self-loops** (`nameOrig == nameDest`).
* **Analysis:** PaySim contains zero trivial recursive edges. Transition and adjacency matrices will not suffer from immediate diagonal self-absorption, ensuring clean PageRank iterations without artificial single-node rank traps.

### **5. Path Traversal & Cycle Feasibility (Critical for Person 4 / Task 3)**

| Metric | Tested Sample / Cap | Observed Result | Topological Status |
| :--- | :--- | :--- | :--- |
| **C2C Transactions** | Full Dataset | **4,211,125** | High liquidity pool |
| **2-Hop Chains ($A \to B \to C$)** | Capped at 2,000 | **2,000 (Hit cap)** | Abundant ($> 2,000$) |
| **Reciprocal Pairs ($A \leftrightarrow B$)** | Capped at 500 | **0** | **Exhaustive 0** across dataset |
| **3-Node Cycles ($A \to B \to C \to A$)** | Capped at 10 | **0** | **Exhaustive 0** across dataset |


> **Critical Note on Cap Interpretation:** Unlike the 2-hop row (which hit its cap of 2,000 early, indicating thousands more exist), the reciprocal-pair and 3-node-cycle rows returned **0 without hitting their caps (500, 10)**. This confirms that these are **complete, exhaustive results across the entire 6.36M rows**, not sampling artifacts.

* **Topological Root Cause:** 
  PaySim models fraud as a system exit sequence rather than a circular flow: an attacker compromises an account, issues a `TRANSFER` to a mule, who immediately issues a `CASH_OUT` to liquidate the balance outside the mobile money network:
  $$\text{Victim} \xrightarrow{\text{TRANSFER}} \text{Mule} \xrightarrow{\text{CASH\_OUT}} \text{Liquidator / Agent}$$
  Because stolen capital leaves the system rather than returning to the victim, **closed 3-node transaction cycles ($A \to B \to C \to A$) do not exist in default PaySim**.

* **Impact on Task 3:** The literal motif query in the proposal:
  ```python
  graph.find("(a)-[e1]->(b); (b)-[e2]->(c); (c)-[e3]->(a)")
  ```
  will return **zero rows** on this dataset. This confirms the concrete risk flagged in Section 7.1/7.2 of the project proposal.

* **Two Concrete Resolution Paths for the Team:**
  1. **Option 1 (Redefine Motif Pattern):** Instead of a circular return-to-origin loop, redefine Task 3 to detect suspicious topological patterns PaySim actually contains—specifically the **2-hop relay pattern** ($A \xrightarrow{\text{TRANSFER}} B \xrightarrow{\text{CASH\_OUT}} C$) or a **fan-in concentration hub** ($\{A_1, A_2, A_3\} \to B$), both of which are documented AML typologies.
  2. **Option 2 (Contingency Switch to AMLSim):** Fall back to the AMLSim dataset per Section 7.2 of the proposal, which explicitly injects circular laundering topologies. This requires adapting the column mapping table (Section 4.1).

  *This is a scope decision for Person 1 (Lead) and Person 4 (Motif Owner) to finalize together early.*

### **6. Fraud Ground Truth & Rule-Based Detection Failure**

| Transaction Type | Total Volume | Fraud Transactions | Fraud Rate (%) | Flagged by System (`isFlaggedFraud`) |
| :--- | :--- | :--- | :--- | :--- |
| **TRANSFER** | 532,909 | **4,097** | **0.7688%** | **16** |
| **CASH_OUT** | 2,237,500 | **4,116** | **0.1840%** | **0** |
| **PAYMENT** | 2,151,495 | 0 | 0.0000% | 0 |
| **CASH_IN** | 1,399,284 | 0 | 0.0000% | 0 |
| **DEBIT** | 41,432 | 0 | 0.0000% | 0 |
| **Total** | **6,362,620** | **8,213** | **0.1291%** | **16 (0.19% recall)** |

* **Detection Miss Rate:** Out of 8,213 fraudulent transactions, the legacy rule-based threshold (`amount > 200,000` via `isFlaggedFraud`) caught only **16 instances**—a **99.81% miss rate** (recall of 0.19%).
* **Business Justification:** This provides empirical proof for the report introduction that static threshold rules fail against modern laundering schemes, establishing the direct motivation for graph-based anomaly detection.


### **7. Cross-Task Impact Matrix**

| Project Task | Downstream Owner | Impact Assessment & Strategic Recommendation |
| :--- | :--- | :--- |
| **Task 1: Graph Construction** | Person 1 & Person 2 | **No impact.** Vertices and edges DataFrames construct cleanly; schema maps 100% to specifications. |
| **Task 2: Degree & PageRank** | Person 3 | **Positive impact.** The sharp imbalance between 6.35M senders and 2.72M receivers creates natural fan-in hubs, producing distinct high-centrality vertices for PageRank and degree distributions. |
| **Task 3: Motif Finding** | Person 4 | **High structural risk.** Closed 3-node cycles evaluate to 0. Requires choosing Option 1 (redefine motif pattern to 2-hop relay / fan-in) or Option 2 (AMLSim fallback). |
| **Task 4: Community Detection (LPA)** | Person 5 & Person 6 | **Low risk.** Label Propagation runs on the underlying undirected graph representation. Dense fan-in structures around the 1,769 intermediate hub nodes will produce non-trivial communities regardless of directed cycle absence. |

